In [1]:
import pandas as pd
import numpy as np

def clean_raw_data(file_path='Cleaned Data.csv'):
    print("=== [STEP 0] Data Cleaning & Sanity Audit ===")
    
    # 1. Load data safely specifying separator
    try:
        df = pd.read_csv(file_path, sep=';')
    except Exception:
        df = pd.read_csv(file_path)
        
    initial_count = len(df)
    print(f"Initial raw record count: {initial_count:,}")

    # 2. Check for missing values (NaNs)
    missing_counts = df.isnull().sum().sum()
    if missing_counts > 0:
        print(f"Warning: Found {missing_counts} missing values. Imputing/Dropping...")
        df.dropna(inplace=True)
    else:
        print("✓ Missing values check: 0 NaNs found.")

    # 3. Check and remove duplicate rows
    duplicates = df.duplicated().sum()
    if duplicates > 0:
        print(f"Removing {duplicates} duplicate transaction rows...")
        df.drop_duplicates(inplace=True)
    else:
        print("✓ Duplicates check: 0 duplicate rows found.")

    # 4. Handle invalid/zero transaction amounts
    zero_amount_count = (df['amount'] <= 0).sum()
    if zero_amount_count > 0:
        print(f"Filtering out {zero_amount_count} invalid/zero transaction amount records...")
        df = df[df['amount'] > 0].copy()
    else:
        print("✓ Amount validity check: All transactions have valid amounts (> 0).")

    # 5. Enforce strict data types for memory optimization and stability
    dtype_mapping = {
        'step': 'int32',
        'type': 'category',
        'amount': 'float64',
        'nameOrig': 'string',
        'oldbalanceOrg': 'float64',
        'newbalanceOrig': 'float64',
        'nameDest': 'string',
        'oldbalanceDest': 'float64',
        'newbalanceDest': 'float64',
        'isFraud': 'int8',
        'isFlaggedFraud': 'int8'
    }
    
    df = df.astype({col: dtype for col, dtype in dtype_mapping.items() if col in df.columns})

    print(f"Cleaned dataset record count: {len(df):,} (Removed {initial_count - len(df)} bad records)")
    print("=== [STEP 0 COMPLETE] Cleaned Data Ready for Step 1 Filtering ===\n")
    
    return df

if __name__ == '__main__':
    df_clean = clean_raw_data('Cleaned Data.csv')
    df_clean.to_csv('cleaned_step0_output.csv', index=False, sep=';')

=== [STEP 0] Data Cleaning & Sanity Audit ===
Initial raw record count: 636,262
✓ Missing values check: 0 NaNs found.
✓ Duplicates check: 0 duplicate rows found.
Filtering out 2 invalid/zero transaction amount records...
Cleaned dataset record count: 636,260 (Removed 2 bad records)
=== [STEP 0 COMPLETE] Cleaned Data Ready for Step 1 Filtering ===



In [2]:
import pandas as pd
import numpy as np

def load_and_preprocess_step1(file_path='Cleaned Data.csv'):
    print("--- [STEP 1] Loading and Preprocessing Data ---")
    
    # 1. Load the raw dataset
    df = pd.read_csv(file_path, sep=';')
    print(f"Initial raw shape: {df.shape}")
    print(f"Total fraud instances in raw data: {df['isFraud'].sum()}")
    
    # 2. Filter strictly for high-risk transaction types
    # (100% of fraud occurs in TRANSFER and CASH_OUT)
    high_risk_types = ['TRANSFER', 'CASH_OUT']
    df_filtered = df[df['type'].isin(high_risk_types)].copy()
    
    print(f"Filtered shape (TRANSFER & CASH_OUT only): {df_filtered.shape}")
    print(f"Fraud instances preserved: {df_filtered['isFraud'].sum()}")
    
    # 3. Time Feature Engineering (1 step = 1 hour)
    df_filtered['hour_of_day'] = df_filtered['step'] % 24
    df_filtered['day_of_week'] = (df_filtered['step'] // 24) % 7
    
    # 4. Accounting Mismatch / Balance Error Features
    df_filtered['errorBalanceOrig'] = (
        df_filtered['oldbalanceOrg'] - df_filtered['amount'] - df_filtered['newbalanceOrig']
    )
    df_filtered['errorBalanceDest'] = (
        df_filtered['oldbalanceDest'] + df_filtered['amount'] - df_filtered['newbalanceDest']
    )
    
    # 5. Entity Type Indicator (Merchant recipient flag)
    df_filtered['isMerchantDest'] = df_filtered['nameDest'].str.startswith('M').astype(int)
    
    # 6. Drop non-predictive ID columns and useless flags
    cols_to_drop = ['nameOrig', 'nameDest', 'isFlaggedFraud']
    df_filtered.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    
    # Verify cleaned state
    print(f"Missing values count:\n{df_filtered.isnull().sum().sum()}")
    print("--- [STEP 1 COMPLETE] Data ready for Visualization & Feature Encoding ---\n")
    
    return df_filtered

# Run Step 1
if __name__ == '__main__':
    df_cleaned = load_and_preprocess_step1('Cleaned Data.csv')
    # Save processed dataframe for Step 2
    df_cleaned.to_csv('step1_output.csv', index=False, sep=';')
    print(df_cleaned.head())

--- [STEP 1] Loading and Preprocessing Data ---
Initial raw shape: (636262, 11)
Total fraud instances in raw data: 849
Filtered shape (TRANSFER & CASH_OUT only): (277116, 11)
Fraud instances preserved: 849
Missing values count:
0
--- [STEP 1 COMPLETE] Data ready for Visualization & Feature Encoding ---

    step      type     amount  oldbalanceOrg  newbalanceOrig  oldbalanceDest  \
4    563  CASH_OUT  246476.67           0.00            0.00      4226850.16   
6    356  CASH_OUT  171545.04       21858.00            0.00            0.00   
7    258  CASH_OUT   94295.16      294254.16       199958.99      2390028.04   
9    399  CASH_OUT  466080.27        2753.00            0.00            0.00   
10   163  CASH_OUT  138265.04       10501.00            0.00            0.00   

    newbalanceDest  isFraud  hour_of_day  day_of_week  errorBalanceOrig  \
4       4473326.82        0           11            2        -246476.67   
6        171545.04        0           20            0        -14

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def generate_step2_eda_dashboard(file_path='Cleaned Data.csv'):
    print("--- [STEP 2] Generating Exploratory Data Analysis (EDA) Visualizations ---")

    # 1. Load Data
    df = pd.read_csv(file_path, sep=';')

    # Set Seaborn theme safely (compatible across all Matplotlib/Seaborn versions)
    sns.set_theme(style='whitegrid')
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # -------------------------------------------------------------
    # Plot 1: Origin Balance Drain Ratio - Legit vs Fraud
    # (amount / (oldbalanceOrg + 1)) - how much of the sender's
    # balance got wiped out in the transaction. Same feature used in
    # Step 3 modeling and the explainability engine, so this panel
    # actually explains why the model flags what it flags.
    # -------------------------------------------------------------
    df['origDrainRatio'] = df['amount'] / (df['oldbalanceOrg'] + 1)
    drain_plot_df = df.copy()
    drain_plot_df['origDrainRatio_clipped'] = drain_plot_df['origDrainRatio'].clip(upper=2)

    sns.boxplot(
        x='isFraud',
        y='origDrainRatio_clipped',
        data=drain_plot_df,
        palette={0: '#1f77b4', 1: '#d62728'},
        ax=axes[0, 0]
    )
    axes[0, 0].set_title('1. Origin Balance Drain Ratio: Legit vs Fraud', fontsize=12, fontweight='bold', pad=10)
    axes[0, 0].set_xlabel('Class (0 = Legitimate, 1 = Fraud)', fontsize=10)
    axes[0, 0].set_ylabel('Drain Ratio (amount / oldbalanceOrg), clipped at 2', fontsize=10)
    axes[0, 0].set_xticks([0, 1])
    axes[0, 0].set_xticklabels(['Legitimate', 'Fraudulent'])

    # -------------------------------------------------------------
    # Plot 2: Transaction Volume & Fraud Counts by Type (Log Scale)
    # Grouped (side-by-side) bars instead of stacked, so the small
    # fraud counts get their own visible bar instead of being crushed
    # into a sliver on top of the much larger legit bar.
    # -------------------------------------------------------------
    type_counts = df.groupby('type')['isFraud'].agg(['count', 'sum']).reset_index()
    type_counts.columns = ['type', 'total', 'fraud']
    type_counts['legit'] = type_counts['total'] - type_counts['fraud']

    x_pos = np.arange(len(type_counts))
    width = 0.35
    axes[0, 1].bar(x_pos - width/2, type_counts['legit'], width, label='Legitimate (0)', color='#1f77b4')
    axes[0, 1].bar(x_pos + width/2, type_counts['fraud'], width, label='Fraudulent (1)', color='#d62728')
    axes[0, 1].set_yscale('log')
    axes[0, 1].set_ylim(bottom=0.5)  # keeps zero-count bars from vanishing off-scale
    axes[0, 1].set_title('2. Transaction Volume & Fraud Counts by Type (Log Scale)', fontsize=12, fontweight='bold', pad=10)
    axes[0, 1].set_xlabel('Transaction Type', fontsize=10)
    axes[0, 1].set_ylabel('Count (Log Scale)', fontsize=10)
    axes[0, 1].set_xticks(x_pos)
    axes[0, 1].set_xticklabels(type_counts['type'])
    axes[0, 1].legend(loc='upper right')

    # Label each fraud bar with its exact count for readability on log scale
    for i, fraud_count in enumerate(type_counts['fraud']):
        if fraud_count > 0:
            axes[0, 1].text(
                x_pos[i] + width/2, fraud_count * 1.3, f'{int(fraud_count)}',
                ha='center', va='bottom', fontsize=8, color='#d62728', fontweight='bold'
            )

    # -------------------------------------------------------------
    # Plot 3: Transaction Amount - Legitimate vs Fraud (Log Scale)
    # -------------------------------------------------------------
    sns.boxplot(
        x='isFraud',
        y='amount',
        data=df,
        palette={0: '#1f77b4', 1: '#d62728'},
        ax=axes[1, 0]
    )
    axes[1, 0].set_yscale('log')
    axes[1, 0].set_title('3. Transaction Amount: Legitimate vs Fraud (Log Scale)', fontsize=12, fontweight='bold', pad=10)
    axes[1, 0].set_xlabel('Class (0 = Legitimate, 1 = Fraud)', fontsize=10)
    axes[1, 0].set_ylabel('Amount (Log Scale)', fontsize=10)
    axes[1, 0].set_xticks([0, 1])
    axes[1, 0].set_xticklabels(['Legitimate', 'Fraudulent'])

    # -------------------------------------------------------------
    # Plot 4: Balance Discrepancy Rate by Class (%)
    # -------------------------------------------------------------
    df['errorBalanceOrig'] = df['oldbalanceOrg'] - df['amount'] - df['newbalanceOrig']
    df['hasDiscrepancy'] = (df['errorBalanceOrig'].abs() > 0.01).astype(int)

    disc_table = pd.crosstab(df['hasDiscrepancy'], df['isFraud'], normalize='columns') * 100
    disc_table = disc_table.rename(index={0: 'No Discrepancy', 1: 'Discrepancy Present'})

    x_pos2 = np.arange(len(disc_table))
    width2 = 0.35
    axes[1, 1].bar(x_pos2 - width2/2, disc_table[0], width2, label='Legitimate', color='#1f77b4')
    axes[1, 1].bar(x_pos2 + width2/2, disc_table[1], width2, label='Fraudulent', color='#d62728')
    axes[1, 1].set_title('4. Balance Discrepancy Rate by Class (%)', fontsize=12, fontweight='bold', pad=10)
    axes[1, 1].set_xlabel('Balance Discrepancy (0 = Matches Math, 1 = Mismatch)', fontsize=10)
    axes[1, 1].set_ylabel('Percentage within Class (%)', fontsize=10)
    axes[1, 1].set_xticks(x_pos2)
    axes[1, 1].set_xticklabels(disc_table.index)
    axes[1, 1].legend(loc='upper right')

    plt.tight_layout()
    output_filename = 'step2_eda_dashboard.png'
    plt.savefig(output_filename, dpi=300)
    plt.close()
    print(f"--- [STEP 2 COMPLETE] Dashboard saved successfully as '{output_filename}' ---\n")

if __name__ == '__main__':
    generate_step2_eda_dashboard('Cleaned Data.csv')

--- [STEP 2] Generating Exploratory Data Analysis (EDA) Visualizations ---
--- [STEP 2 COMPLETE] Dashboard saved successfully as 'step2_eda_dashboard.png' ---



In [ ]:
import numpy as np
import pandas as pd

print("=== [STEP 3] Feature Creation, Transformation & Selection ===")

# 1. LOAD CLEANED DATA
print("\n1. Loading dataset...")
df = pd.read_csv('Cleaned Data.csv', sep=';')

# 2. FEATURE CREATION & FEATURE ADDITION
print("2. Creating engineered features (accounting discrepancies, merchant indicator, balance-drain ratios)...")
# Calculate accounting error discrepancies
df['errorBalanceOrig'] = (
    df['oldbalanceOrg'] - df['amount'] - df['newbalanceOrig']
)
df['errorBalanceDest'] = (
    df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']
)

# Boolean indicator for Merchant targets (destination names starting with 'M')
df['isMerchantDest'] = (
    df['nameDest'].astype(str).str.startswith('M').astype(int)
)

# --- NEW: balance-drain features (non-time) ---------------------------
# These directly encode the classic fraud "signature": the origin account
# gets emptied (or near-emptied) in a single transaction. This is a much
# stronger, more stable signal for a linear model than raw balances alone,
# since it's already a ratio/flag rather than a large skewed number.
df['origDrainRatio'] = (
    df['amount'] / (df['oldbalanceOrg'] + 1)
)  # ~1.0 means "moved (almost) the entire origin balance"
df['isOrigDrained'] = (
    (df['newbalanceOrig'] == 0) & (df['oldbalanceOrg'] > 0)
).astype(int)


# 3. FEATURE TRANSFORMATION (LOG TRANSFORMATION)
print("3. Applying log1p transformations & One-Hot Encoding...")
# Log-transform skewed financial features: log(1 + x)
log_cols = [
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'oldbalanceDest',
    'newbalanceDest',
]
for col in log_cols:
    df[f'{col}_log'] = np.log1p(df[col])

# One-Hot Encode categorical transaction type
df = pd.get_dummies(df, columns=['type'], drop_first=True)

# 4. FEATURE SELECTION (DROPPING NOISY / IDENTIFIER COLS)
print("4. Selecting predictive features and setting target variable...")
drop_cols = ['nameOrig', 'nameDest', 'isFlaggedFraud', 'isFraud'] + log_cols

X = df.drop(columns=drop_cols)
y = df['isFraud']

# 5. SUMMARY OUTPUT
print('\n--- STEP 3 FEATURE PREPARATION COMPLETE ---')
print(f'Feature Matrix (X) Shape: {X.shape}')
print(f'Target Variable (y) Shape: {y.shape}')

print('\nFinal Selected Features for Model Training:')
for i, col in enumerate(X.columns, 1):
    print(f'{i:2d}. {col}')

# 6. SAVE PREPROCESSED DATASET (Optional for separate model script)
output_file = 'features_ready_for_modeling.csv'
df_features = pd.concat([X, y], axis=1)
df_features.to_csv(output_file, index=False, sep=';')
print(f"\n✓ Saved prepared feature set to '{output_file}'!")


=== [STEP 3] Feature Creation, Transformation & Selection ===

1. Loading dataset...
2. Creating engineered features (accounting discrepancies, merchant indicator, balance-drain ratios)...
3. Applying log1p transformations & One-Hot Encoding...
4. Selecting predictive features and setting target variable...

--- STEP 3 FEATURE PREPARATION COMPLETE ---
Feature Matrix (X) Shape: (636262, 15)
Target Variable (y) Shape: (636262,)

Final Selected Features for Model Training:
 1. step
 2. errorBalanceOrig
 3. errorBalanceDest
 4. isMerchantDest
 5. origDrainRatio
 6. isOrigDrained
 7. amount_log
 8. oldbalanceOrg_log
 9. newbalanceOrig_log
10. oldbalanceDest_log
11. newbalanceDest_log
12. type_CASH_OUT
13. type_DEBIT
14. type_PAYMENT
15. type_TRANSFER

✓ Saved prepared feature set to 'features_ready_for_modeling.csv'!


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, f1_score, precision_score, recall_score
)
from imblearn.over_sampling import SMOTE

print("=== [MODEL 1 - PRIMARY / PRODUCTION MODEL] Logistic Regression (Tuned, SMOTE-Balanced) ===")

# 1. LOAD PREPARED FEATURE SET
print("1. Loading preprocessed feature data...")
df = pd.read_csv('features_ready_for_modeling.csv', sep=';')

X = df.drop(columns=['isFraud'])
y = df['isFraud']

# 2. STRATIFIED TRAIN / TEST SPLIT (70/30)
# The TEST split is left at its real, imbalanced ratio - only the TRAINING
# split gets balanced/tuned against.
print("2. Splitting data into 70% Train and 30% Test sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"   Original Training Class 0 (Legit) Count: {sum(y_train == 0):,}")
print(f"   Original Training Class 1 (Fraud) Count: {sum(y_train == 1):,}")

# 3. BALANCE THE TRAINING DATA WITH SMOTE
print("3. Balancing training data with SMOTE...")
smote = SMOTE(sampling_strategy="auto", random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"   Balanced Training Class 0 (Legit) Count: {sum(y_train_bal == 0):,}")
print(f"   Balanced Training Class 1 (Fraud) Count: {sum(y_train_bal == 1):,}")

# 4. SCALE FEATURES (Logistic Regression is sensitive to feature scale)
print("4. Scaling features using StandardScaler...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# 5. HYPERPARAMETER TUNING (model improvement)
# 5-fold cross-validated grid search over regularization strength C, scored
# on ROC-AUC. This replaces the old "just pick lbfgs defaults" approach
# with an actual search for the best-generalizing regularization strength,
# instead of guessing.
print("5. Tuning regularization strength (C) via 5-fold cross-validated GridSearchCV...")
param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42),
    param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
)
grid.fit(X_train_scaled, y_train_bal)

print(f"   Best C: {grid.best_params_['C']}  (CV ROC-AUC: {grid.best_score_:.4f})")
log_reg = grid.best_estimator_

# NOTE: class_weight='balanced' is intentionally NOT combined with SMOTE -
# SMOTE already makes the training set 50/50, so also reweighting classes
# would double-correct and tank precision.

# 6. EVALUATION & PREDICTIONS (standard 0.5 threshold, same metric block
# used by every other model cell so results are directly comparable)
print("\n6. Evaluating performance on the untouched test set...\n")
y_pred = log_reg.predict(X_test_scaled)
y_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

print("="*55)
print("       LOGISTIC REGRESSION METRICS (threshold=0.5)       ")
print("="*55)
print(f"Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_proba):.4f}")
# PR-AUC (average precision) is a more honest summary than ROC-AUC when
# the positive class (fraud) is rare - it isn't inflated by the huge
# number of easy true negatives.
print(f"PR-AUC    : {average_precision_score(y_test, y_proba):.4f}")
print("="*55)

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(pd.DataFrame(cm, index=['Actual Legit', 'Actual Fraud'], columns=['Pred Legit', 'Pred Fraud']))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraudulent']))

# 7. HIGH-PRECISION THRESHOLD SCAN (kept from the original design - useful
# for a compliance team deciding how aggressive the alert threshold should be)
high_thresholds = [0.85, 0.90, 0.95, 0.98, 0.99]
print("\n" + "="*55)
print("       EVALUATING HIGH PRECISION THRESHOLDS (bonus)      ")
print("="*55)
for t in high_thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn, fp, fn, tp = cm_t.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f"\n--- THRESHOLD: {t:.2f} ---")
    print(f"Precision: {precision:.4f}  |  Recall: {recall:.4f}  ({tp}/{tp+fn} caught)")
    print(f"False Positives (False Alarms): {fp}")
    print(f"False Negatives (Missed Cases): {fn}")

# 8. COEFFICIENT CHART - visualizes the ACTUAL tuned production model and,
# unlike an importance score, also shows the DIRECTION of each effect
# (green = raises fraud odds, red = lowers it).
print("\n8. Generating Logistic Regression coefficient chart...")
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': log_reg.coef_[0]
}).sort_values('coefficient', key=abs, ascending=True)

colors = ['#d62728' if c < 0 else '#2ca02c' for c in coef_df['coefficient']]

plt.figure(figsize=(10, 8))
plt.barh(coef_df['feature'], coef_df['coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title(f'Logistic Regression Coefficients (C={grid.best_params_["C"]}, SMOTE-Balanced)', fontsize=13, fontweight='bold')
plt.xlabel('Standardized coefficient  —  green = raises fraud odds, red = lowers it')
plt.tight_layout()
output_filename = 'step4_logreg_coefficients.png'
plt.savefig(output_filename, dpi=300)
plt.close()
print(f"✓ Coefficient chart saved as '{output_filename}' (replaces step4_feature_importance.png)")

# Keep fitted objects available for the Explainability Engine cell below
final_pipeline_scaler = scaler
final_pipeline_model = log_reg


=== [MODEL 1 - PRIMARY / PRODUCTION MODEL] Logistic Regression (Tuned, SMOTE-Balanced) ===
1. Loading preprocessed feature data...
2. Splitting data into 70% Train and 30% Test sets...
   Original Training Class 0 (Legit) Count: 444,789
   Original Training Class 1 (Fraud) Count: 594
3. Balancing training data with SMOTE...
   Balanced Training Class 0 (Legit) Count: 444,789
   Balanced Training Class 1 (Fraud) Count: 444,789
4. Scaling features using StandardScaler...
5. Tuning regularization strength (C) via 5-fold cross-validated GridSearchCV...
   Best C: 100  (CV ROC-AUC: 0.9993)

6. Evaluating performance on the untouched test set...

       LOGISTIC REGRESSION METRICS (threshold=0.5)       
Precision : 0.0785
Recall    : 0.9922
F1-Score  : 0.1455
ROC-AUC   : 0.9988
PR-AUC    : 0.6706

Confusion Matrix:
              Pred Legit  Pred Fraud
Actual Legit      187655        2969
Actual Fraud           2         253

Classification Report:
              precision    recall  f1-score 

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, precision_score, recall_score

print("=== [MODEL 2] Decision Tree Classifier ===")

# 1. LOAD PREPARED FEATURE SET
print("1. Loading preprocessed feature data...")
df = pd.read_csv('features_ready_for_modeling.csv', sep=';')

X = df.drop(columns=['isFraud'])
y = df['isFraud']

# 2. STRATIFIED TRAIN / TEST SPLIT (70/30)
print("2. Splitting data into 70% Train and 30% Test sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 3. INITIALIZE AND FIT MODEL
print("3. Fitting Decision Tree model (max_depth=10 to control overfitting)...")
dt_clf = DecisionTreeClassifier(
    max_depth=10, 
    random_state=42, 
    criterion='gini'
)
dt_clf.fit(X_train, y_train)

# 4. EVALUATION & PREDICTIONS
print("4. Evaluating performance on the test set...\n")
y_pred = dt_clf.predict(X_test)
y_proba = dt_clf.predict_proba(X_test)[:, 1]

# Display Metrics
print("="*55)
print("              DECISION TREE METRICS              ")
print("="*55)
print(f"Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_proba):.4f}")
print("="*55)

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(pd.DataFrame(cm, index=['Actual Legit', 'Actual Fraud'], columns=['Pred Legit', 'Pred Fraud']))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraudulent']))

=== [MODEL 2] Decision Tree Classifier ===
1. Loading preprocessed feature data...
2. Splitting data into 70% Train and 30% Test sets...
3. Fitting Decision Tree model (max_depth=10 to control overfitting)...
4. Evaluating performance on the test set...

              DECISION TREE METRICS              
Precision : 0.9961
Recall    : 1.0000
F1-Score  : 0.9980
ROC-AUC   : 1.0000

Confusion Matrix:
              Pred Legit  Pred Fraud
Actual Legit      190623           1
Actual Fraud           0         255

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00    190624
  Fraudulent       1.00      1.00      1.00       255

    accuracy                           1.00    190879
   macro avg       1.00      1.00      1.00    190879
weighted avg       1.00      1.00      1.00    190879



In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, precision_score, recall_score

print("=== [MODEL 3] Random Forest Classifier ===")

# 1. LOAD PREPARED FEATURE SET
print("1. Loading preprocessed feature data...")
df = pd.read_csv('features_ready_for_modeling.csv', sep=';')

X = df.drop(columns=['isFraud'])
y = df['isFraud']

# 2. STRATIFIED TRAIN / TEST SPLIT (70/30)
print("2. Splitting data into 70% Train and 30% Test sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 3. INITIALIZE AND FIT MODEL
print("3. Fitting Random Forest model (50 trees, max_depth=15, multi-core)...")
rf_clf = RandomForestClassifier(
    n_estimators=50, 
    max_depth=15, 
    random_state=42, 
    n_jobs=-1
)
rf_clf.fit(X_train, y_train)

# 4. EVALUATION & PREDICTIONS
print("4. Evaluating performance on the test set...\n")
y_pred = rf_clf.predict(X_test)
y_proba = rf_clf.predict_proba(X_test)[:, 1]

# Display Metrics
print("="*55)
print("              RANDOM FOREST METRICS              ")
print("="*55)
print(f"Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_proba):.4f}")
print("="*55)

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(pd.DataFrame(cm, index=['Actual Legit', 'Actual Fraud'], columns=['Pred Legit', 'Pred Fraud']))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraudulent']))

=== [MODEL 3] Random Forest Classifier ===
1. Loading preprocessed feature data...
2. Splitting data into 70% Train and 30% Test sets...
3. Fitting Random Forest model (50 trees, max_depth=15, multi-core)...
4. Evaluating performance on the test set...

              RANDOM FOREST METRICS              
Precision : 1.0000
Recall    : 1.0000
F1-Score  : 1.0000
ROC-AUC   : 1.0000

Confusion Matrix:
              Pred Legit  Pred Fraud
Actual Legit      190624           0
Actual Fraud           0         255

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00    190624
  Fraudulent       1.00      1.00      1.00       255

    accuracy                           1.00    190879
   macro avg       1.00      1.00      1.00    190879
weighted avg       1.00      1.00      1.00    190879



In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, precision_score, recall_score

print("=== [MODEL 4] Gradient Boosting Classifier ===")

# 1. LOAD PREPARED FEATURE SET
print("1. Loading preprocessed feature data...")
df = pd.read_csv('features_ready_for_modeling.csv', sep=';')

X = df.drop(columns=['isFraud'])
y = df['isFraud']

# 2. STRATIFIED TRAIN / TEST SPLIT (70/30)
print("2. Splitting data into 70% Train and 30% Test sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 3. INITIALIZE AND FIT MODEL
print("3. Fitting HistGradientBoosting model (50 trees, fast histogram binning)...")
gb_clf = HistGradientBoostingClassifier(
    max_iter=50, 
    max_depth=5, 
    random_state=42
)
gb_clf.fit(X_train, y_train)

# 4. EVALUATION & PREDICTIONS
print("4. Evaluating performance on the test set...\n")
y_pred = gb_clf.predict(X_test)
y_proba = gb_clf.predict_proba(X_test)[:, 1]

# Display Metrics
print("="*55)
print("            GRADIENT BOOSTING METRICS            ")
print("="*55)
print(f"Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_proba):.4f}")
print("="*55)

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(pd.DataFrame(cm, index=['Actual Legit', 'Actual Fraud'], columns=['Pred Legit', 'Pred Fraud']))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraudulent']))

=== [MODEL 4] Gradient Boosting Classifier ===
1. Loading preprocessed feature data...
2. Splitting data into 70% Train and 30% Test sets...
3. Fitting HistGradientBoosting model (50 trees, fast histogram binning)...
4. Evaluating performance on the test set...

            GRADIENT BOOSTING METRICS            
Precision : 0.9961
Recall    : 1.0000
F1-Score  : 0.9980
ROC-AUC   : 1.0000

Confusion Matrix:
              Pred Legit  Pred Fraud
Actual Legit      190623           1
Actual Fraud           0         255

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00    190624
  Fraudulent       1.00      1.00      1.00       255

    accuracy                           1.00    190879
   macro avg       1.00      1.00      1.00    190879
weighted avg       1.00      1.00      1.00    190879



In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, precision_score, recall_score

print("=== [MODEL 5] K-Nearest Neighbors (KNN) Classifier ===")

# 1. LOAD PREPARED FEATURE SET
print("1. Loading preprocessed feature data...")
df = pd.read_csv('features_ready_for_modeling.csv', sep=';')

X = df.drop(columns=['isFraud'])
y = df['isFraud']

# 2. STRATIFIED TRAIN / TEST SPLIT (70/30)
print("2. Splitting data into 70% Train and 30% Test sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 3. FEATURE SCALING (Crucial for distance-based models)
print("3. Scaling features using StandardScaler...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Subsampling for fast distance evaluation (prevents O(N^2) memory lockup)
np.random.seed(42)
train_sample_idx = np.random.choice(len(X_train_scaled), 50000, replace=False)
test_sample_idx = np.random.choice(len(X_test_scaled), 15000, replace=False)

X_tr_sub = X_train_scaled[train_sample_idx]
y_tr_sub = y_train.iloc[train_sample_idx]
X_te_sub = X_test_scaled[test_sample_idx]
y_te_sub = y_test.iloc[test_sample_idx]

# 4. INITIALIZE AND FIT MODEL
print("4. Fitting KNN model (k=5, kd_tree algorithm, multi-core)...")
knn_clf = KNeighborsClassifier(
    n_neighbors=5, 
    algorithm='kd_tree', 
    n_jobs=-1
)
knn_clf.fit(X_tr_sub, y_tr_sub)

# 5. EVALUATION & PREDICTIONS
print("5. Evaluating performance on the test set...\n")
y_pred = knn_clf.predict(X_te_sub)
y_proba = knn_clf.predict_proba(X_te_sub)[:, 1]

# Display Metrics
print("="*55)
print("             K-NEAREST NEIGHBORS (KNN) METRICS             ")
print("="*55)
print(f"Precision : {precision_score(y_te_sub, y_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_te_sub, y_pred, zero_division=0):.4f}")
print(f"F1-Score  : {f1_score(y_te_sub, y_pred, zero_division=0):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_te_sub, y_proba):.4f}")
print("="*55)

print("\nConfusion Matrix:")
cm = confusion_matrix(y_te_sub, y_pred)
print(pd.DataFrame(cm, index=['Actual Legit', 'Actual Fraud'], columns=['Pred Legit', 'Pred Fraud']))

print("\nClassification Report:")
print(classification_report(y_te_sub, y_pred, target_names=['Legitimate', 'Fraudulent']))

=== [MODEL 5] K-Nearest Neighbors (KNN) Classifier ===
1. Loading preprocessed feature data...
2. Splitting data into 70% Train and 30% Test sets...
3. Scaling features using StandardScaler...
4. Fitting KNN model (k=5, kd_tree algorithm, multi-core)...
5. Evaluating performance on the test set...



c:\Users\Admin\anaconda3\lib\site-packages\sklearn\neighbors\_classification.py:228: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  mode, _ = stats.mode(_y[neigh_ind, k], axis=1)


             K-NEAREST NEIGHBORS (KNN) METRICS             
Precision : 0.8333
Recall    : 0.5882
F1-Score  : 0.6897
ROC-AUC   : 0.8822

Confusion Matrix:
              Pred Legit  Pred Fraud
Actual Legit       14981           2
Actual Fraud           7          10

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     14983
  Fraudulent       0.83      0.59      0.69        17

    accuracy                           1.00     15000
   macro avg       0.92      0.79      0.84     15000
weighted avg       1.00      1.00      1.00     15000



In [11]:
import pandas as pd
import numpy as np

print("=== [EXPLAINABILITY ENGINE] Generating Reason Codes for ALL Flagged Transactions ===")
print("(Now powered by HistGradientBoosting.)")

# 0. ENSURE gb_clf EXISTS (retrain if the [MODEL 4] cell hasn't been run yet)
if 'gb_clf' not in globals():
    print("0. 'gb_clf' not found in memory - training it now...")
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import HistGradientBoostingClassifier

    df_train = pd.read_csv('features_ready_for_modeling.csv', sep=';')
    X_train_full = df_train.drop(columns=['isFraud'])
    y_train_full = df_train['isFraud']

    X_train, X_test, y_train, y_test = train_test_split(
        X_train_full, y_train_full, test_size=0.3, random_state=42, stratify=y_train_full
    )

    gb_clf = HistGradientBoostingClassifier(max_iter=50, max_depth=5, random_state=42)
    gb_clf.fit(X_train, y_train)
    print("   ✓ gb_clf trained and ready.")

# 1. LOAD PREPARED FEATURE DATA
print("1. Loading dataset...")
df = pd.read_csv('features_ready_for_modeling.csv', sep=';')

X = df.drop(columns=['isFraud'])
y = df['isFraud']

# 2. SCORE THE ENTIRE DATASET USING THE TRAINED HISTGRADIENTBOOSTING MODEL
print("2. Scoring all records across the full dataset...")
df['predicted_fraud'] = gb_clf.predict(X)
df['fraud_probability'] = gb_clf.predict_proba(X)[:, 1]

flagged_df = df[df['predicted_fraud'] == 1].copy()
print(f"-> Total transactions flagged as fraud: {len(flagged_df):,}")

# 3. RULE-BASED REASON CODE GENERATOR
def generate_fraud_reasons(row):
    reasons = []

    if row['newbalanceOrig_log'] == 0 and row['oldbalanceOrg_log'] > 0:
        reasons.append("ACCOUNT_DRAIN: Complete drainage of origin account balance to zero.")

    if row.get('origDrainRatio', 0) >= 0.95:
        reasons.append("HIGH_DRAIN_RATIO: Transaction moved 95%+ of the origin account balance.")

    if abs(row['errorBalanceOrig']) > 0.01:
        reasons.append("BALANCE_DISCREPANCY: Sender balance math does not align with transfer amount.")

    hour = row['step'] % 24
    if hour in [3, 4, 5]:
        reasons.append("HIGH_RISK_WINDOW: Initiated during vulnerable late-night window (03:00 - 05:00 AM), derived from step.")

    if row['amount_log'] > np.log1p(200000):
        reasons.append("HIGH_VALUE_TRANSFER: Transaction volume exceeds threshold (>200,000).")

    if row.get('type_TRANSFER', 0) == 1 or row.get('type_CASH_OUT', 0) == 1:
        reasons.append("HIGH_RISK_CHANNEL: Executed via high-risk channel (TRANSFER / CASH_OUT).")

    return " | ".join(reasons) if reasons else "BEHAVIORAL_ANOMALY: Flagged by HistGradientBoosting on a combination of features."

# 4. APPLY REASON GENERATOR TO ALL FLAGGED TRANSACTIONS
print("3. Generating audit reason codes for all flagged transactions...")
flagged_df['Reason_Codes'] = flagged_df.apply(generate_fraud_reasons, axis=1)

# 5. EXPORT FULL AUDIT REPORT
output_csv = 'all_flagged_transactions_with_reasons.csv'
export_cols = ['step', 'amount_log', 'errorBalanceOrig', 'origDrainRatio', 'fraud_probability', 'isFraud', 'Reason_Codes']

flagged_df[export_cols].to_csv(output_csv, index=True, index_label='Transaction_ID', sep=';')

print(f"\n✓ Successfully exported ALL flagged transactions to '{output_csv}'!")

# 6. DISPLAY SAMPLE LOG
print("\n" + "="*85)
print("                   SAMPLE AUDIT LOG (FIRST 5 FLAGGED ROWS)                   ")
print("="*85)
for idx, row in flagged_df[export_cols].head(5).iterrows():
    print(f"ID: {idx:7d} | Risk Prob: {row['fraud_probability']:.2f} | Reasons: {row['Reason_Codes']}")
print("="*85 + "\n")

=== [EXPLAINABILITY ENGINE] Generating Reason Codes for ALL Flagged Transactions ===
(Now powered by HistGradientBoosting.)
0. 'gb_clf' not found in memory - training it now...
   ✓ gb_clf trained and ready.
1. Loading dataset...
2. Scoring all records across the full dataset...
-> Total transactions flagged as fraud: 850
3. Generating audit reason codes for all flagged transactions...

✓ Successfully exported ALL flagged transactions to 'all_flagged_transactions_with_reasons.csv'!

                   SAMPLE AUDIT LOG (FIRST 5 FLAGGED ROWS)                   
ID:     207 | Risk Prob: 1.00 | Reasons: ACCOUNT_DRAIN: Complete drainage of origin account balance to zero. | HIGH_DRAIN_RATIO: Transaction moved 95%+ of the origin account balance. | HIGH_RISK_CHANNEL: Executed via high-risk channel (TRANSFER / CASH_OUT).
ID:     256 | Risk Prob: 1.00 | Reasons: ACCOUNT_DRAIN: Complete drainage of origin account balance to zero. | HIGH_DRAIN_RATIO: Transaction moved 95%+ of the origin account ba

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

print("=== [BONUS VISUAL] Class Balance Before vs. After SMOTE ===")

# Self-contained: reloads data so this cell can run independently of the
# [MODEL 1] Logistic Regression cell above.
df = pd.read_csv('features_ready_for_modeling.csv', sep=';')
X = df.drop(columns=['isFraud'])
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

before_counts = y_train.value_counts().sort_index()

smote = SMOTE(sampling_strategy="auto", random_state=42)
_, y_train_bal = smote.fit_resample(X_train, y_train)
after_counts = pd.Series(y_train_bal).value_counts().sort_index()

print(f"Original Training Class 0 (Legit) Count: {before_counts[0]:,}")
print(f"Original Training Class 1 (Fraud) Count: {before_counts[1]:,}")
print(f"Balanced Training Class 0 (Legit) Count: {after_counts[0]:,}")
print(f"Balanced Training Class 1 (Fraud) Count: {after_counts[1]:,}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].bar(['Legitimate', 'Fraudulent'], before_counts.values, color=['#1f77b4', '#d62728'])
axes[0].set_title('Training Set BEFORE SMOTE')
axes[0].set_ylabel('Transaction Count')

axes[1].bar(['Legitimate', 'Fraudulent'], after_counts.values, color=['#1f77b4', '#d62728'])
axes[1].set_title('Training Set AFTER SMOTE')

plt.tight_layout()
output_filename = 'step4b_class_balance.png'
plt.savefig(output_filename, dpi=300)
plt.close()
print(f"\n✓ Class balance chart saved as '{output_filename}'")


=== [BONUS VISUAL] Class Balance Before vs. After SMOTE ===
Original Training Class 0 (Legit) Count: 444,789
Original Training Class 1 (Fraud) Count: 594
Balanced Training Class 0 (Legit) Count: 444,789
Balanced Training Class 1 (Fraud) Count: 444,789

✓ Class balance chart saved as 'step4b_class_balance.png'
